In [20]:
import numpy as np 
import pandas as pd

In [21]:
df_calc_train = pd.read_csv("calc_case_description_train_set.csv")
df_mass_train = pd.read_csv("mass_case_description_train_set.csv")
df_calc_test = pd.read_csv("calc_case_description_test_set.csv")
df_mass_test = pd.read_csv("mass_case_description_test_set.csv")

In [22]:
print("CALC TRAIN SHAPE:", df_calc_train.shape)
print("MASS TRAIN SHAPE:", df_mass_train.shape)
print("CALC TEST SHAPE:", df_calc_test.shape)
print("MASS TEST SHAPE:", df_mass_test.shape)

CALC TRAIN SHAPE: (1546, 14)
MASS TRAIN SHAPE: (1318, 14)
CALC TEST SHAPE: (326, 14)
MASS TEST SHAPE: (378, 14)


3568 rows added together between all 4 different csv files

In [23]:
df_calc_train["dataset"] = "calc_train"
df_mass_train["dataset"] = "mass_train"

df_train = pd.concat([df_calc_train, df_mass_train], ignore_index=True)

In [24]:
print("\nCC vs MLO counts:")
print(df_train["image view"].value_counts())


CC vs MLO counts:
image view
MLO    1518
CC     1346
Name: count, dtype: int64


In [25]:
df_calc_test["dataset"] = "calc_test"
df_mass_test["dataset"] = "mass_test"

df = pd.concat([df_train, df_mass_test, df_calc_test], ignore_index=True)

In [28]:
print("\nCC vs MLO counts:")
print(df["image view"].value_counts())
df.head()


CC vs MLO counts:
image view
MLO    1896
CC     1672
Name: count, dtype: int64


,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path,dataset,breast_density,mass shape,mass margins
0,P_00005,3.0,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.959...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9...,calc_train,NaN,NaN,NaN
1,P_00005,3.0,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.95...,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1....,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1....,calc_train,NaN,NaN,NaN
2,P_00007,4.0,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.95...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.95...,calc_train,NaN,NaN,NaN
3,P_00007,4.0,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,Calc-Training_P_00007_LEFT_MLO/1.3.6.1.4.1.959...,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9...,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9...,calc_train,NaN,NaN,NaN
4,P_00008,1.0,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590...,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.95...,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.95...,calc_train,NaN,NaN,NaN


In [29]:
print("\nBy dataset:")
print(pd.crosstab(df["dataset"], df["image view"]))


By dataset:
image view   CC  MLO
dataset             
calc_test   149  177
calc_train  739  807
mass_test   177  201
mass_train  607  711


In [31]:
print("\nTotal patients:", df["patient_id"].nunique())

cc_patients = df[df["image view"] == "CC"]["patient_id"].nunique()
mlo_patients = df[df["image view"] == "MLO"]["patient_id"].nunique()

print("Patients with CC:", cc_patients)
print("Patients with MLO:", mlo_patients)


Total patients: 1566
Patients with CC: 1312
Patients with MLO: 1473


In [33]:
grouped = df.groupby(["patient_id", "left or right breast"])["image view"].apply(set)

def has_pair(views):
    return "CC" in views and "MLO" in views

pairs = grouped[grouped.apply(has_pair)]

print("Number of CC+MLO breast pairs:", len(pairs))

Number of CC+MLO breast pairs: 1324


In [34]:
patients_with_pairs = pairs.reset_index()["patient_id"].nunique()
print("Patients with ≥1 CC+MLO pair:", patients_with_pairs)

Patients with ≥1 CC+MLO pair: 1219


In [36]:
patient_sides = df.groupby("patient_id")["left or right breast"].unique()

both_sides = patient_sides.apply(lambda x: set(x) >= {"LEFT", "RIGHT"})

print("Patients with both LEFT and RIGHT:", both_sides.sum())

Patients with both LEFT and RIGHT: 142


In [38]:
full_coverage = []

for patient, group in df.groupby("patient_id"):
    ok = True
    for side in ["LEFT", "RIGHT"]:
        subset = group[group["left or right breast"] == side]["image view"].unique()
        if not ("CC" in subset and "MLO" in subset):
            ok = False
            break
    if ok:
        full_coverage.append(patient)

print("Patients with FULL coverage (L+R and CC+MLO each):", len(full_coverage))

Patients with FULL coverage (L+R and CC+MLO each): 105


In [39]:
df["split"] = df["dataset"].apply(
    lambda x: "train" if "train" in x else ("test" if "test" in x else "other")
)

In [44]:
patient_splits = df.groupby("patient_id")["split"].unique()

leaky_patients = patient_splits[
    patient_splits.apply(lambda x: len(set(x)) > 1)
]

print("Patients in BOTH train and test:", len(leaky_patients))

Patients in BOTH train and test: 31


In [45]:
breast_splits = df.groupby("breast_id")["split"].unique()

leaky_breasts = breast_splits[
    breast_splits.apply(lambda x: len(set(x)) > 1)
]

print("Breasts appearing in BOTH train and test:", len(leaky_breasts))

Breasts appearing in BOTH train and test: 19


In [46]:
example = leaky_breasts.index[0]

print(df[df["breast_id"] == example][
    ["patient_id", "left or right breast", "image view", "dataset"]
])

     patient_id left or right breast image view     dataset
29      P_00016                 LEFT         CC  calc_train
30      P_00016                 LEFT        MLO  calc_train
2864    P_00016                 LEFT         CC   mass_test
2865    P_00016                 LEFT        MLO   mass_test
